# QRC Progress Ablation: Initial QRC → Final QRC → ESN Reference

This notebook tests whether the regime-warning layer can be used as a compact progress story.

It compares:

1. optional initial QRC prototype, if a row-level prediction table is available;
2. final QRC prototype;
3. ESN classical reservoir reference.

Use this as a storytelling/ablation figure only. If the initial QRC table is unavailable or visually weak, keep the cleaner QRC-vs-ESN comparison plot instead.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == 'notebooks':
    os.chdir('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.evaluation.metrics import evaluate_volatility_forecast

out_table_dir = Path('results/tables')
out_fig_dir = Path('results/figures')
out_table_dir.mkdir(parents=True, exist_ok=True)
out_fig_dir.mkdir(parents=True, exist_ok=True)

## 1. Build the same test-event timeline

In [ ]:
target = 'future_rv_20d'

def make_stress_features(frame):
    out = frame.copy().reset_index(drop=True)
    out['date'] = pd.to_datetime(out['date'])
    out['spy_selloff_stress'] = -out['spy_log_return']
    out['drawdown_stress'] = -out['spy_drawdown_20d'] if out['spy_drawdown_20d'].median() < 0 else out['spy_drawdown_20d']
    out['volume_stress'] = out['spy_log_volume_change'].abs()
    return out

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)
splits = {k: make_stress_features(v) for k, v in splits.items()}
train, test = splits['train'], splits['test']

thresholds = {
    'high_vol_q80': float(train[target].quantile(0.80)),
    'extreme_vol_q90': float(train[target].quantile(0.90)),
    'crisis_candidate_q95': float(train[target].quantile(0.95)),
}

timeline = test.copy().reset_index(drop=True)
timeline['future_high_vol_event_q80'] = timeline[target] > thresholds['high_vol_q80']
timeline['future_extreme_vol_event_q90'] = timeline[target] > thresholds['extreme_vol_q90']
timeline['future_crisis_candidate_q95'] = timeline[target] > thresholds['crisis_candidate_q95']

def q(col, quantile):
    return float(train[col].quantile(quantile))

timeline['direction_tag'] = 'neutral'
timeline.loc[timeline['spy_log_return'] >= q('spy_log_return', 0.70), 'direction_tag'] = 'rally'
timeline.loc[timeline['spy_log_return'] <= q('spy_log_return', 0.30), 'direction_tag'] = 'selloff'

flag_specs = {
    'flag_realized_vol': ('rv_20d', 0.80),
    'flag_vix_level': ('vix_close', 0.80),
    'flag_vix_range': ('vix_log_hl_range', 0.80),
    'flag_drawdown': ('drawdown_stress', 0.80),
    'flag_selloff': ('spy_selloff_stress', 0.80),
    'flag_volume': ('volume_stress', 0.80),
}
for flag, (col, quantile) in flag_specs.items():
    timeline[flag] = timeline[col] > q(col, quantile)

events = ['future_high_vol_event_q80', 'future_extreme_vol_event_q90', 'future_crisis_candidate_q95']
timeline[['date', target] + events].head()

## 2. Load available prediction tables

In [ ]:
# Optional initial QRC candidates. Add another path here if your first-QRC row-level export used a different name.
initial_qrc_candidates = [
    out_table_dir / 'phase2_qrc_anchor_snapshot_predictions.csv',
    out_table_dir / 'phase2_qrc_initial_predictions.csv',
    out_table_dir / 'phase2_qrc_first_predictions.csv',
]

required_tables = {
    'QRC final calibrated prototype': out_table_dir / 'phase2_qrc_final_encoding_readout_predictions.csv',
    'ESN classical reservoir reference': out_table_dir / 'phase2_esn_predictions.csv',
}

available = []
for label, path in required_tables.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing required table: {path}')
    available.append((label, path))

initial_path = next((p for p in initial_qrc_candidates if p.exists()), None)
if initial_path is not None:
    available.insert(0, ('QRC initial prototype', initial_path))
    print(f'Loaded optional initial QRC table: {initial_path}')
else:
    print('No row-level initial QRC prediction table found. The notebook will compare final QRC vs ESN only.')
    print('To include the first QRC, export a table with columns date, actual_future_rv_20d, and one prediction column containing pred in the name.')

available

In [ ]:
def load_prediction_table(label, path):
    tbl = pd.read_csv(path)
    tbl['date'] = pd.to_datetime(tbl['date'])
    pred_cols = [c for c in tbl.columns if 'pred' in c.lower()]
    if not pred_cols:
        raise ValueError(f'No prediction column found in {path}. Columns: {list(tbl.columns)}')
    pred_col = pred_cols[0]
    actual_candidates = [c for c in tbl.columns if c in ['actual_future_rv_20d', 'future_rv_20d', target]]
    if not actual_candidates:
        raise ValueError(f'No actual target column found in {path}. Columns: {list(tbl.columns)}')
    actual_col = actual_candidates[0]
    return tbl[['date', actual_col, pred_col]].rename(columns={actual_col: 'actual_future_rv_20d', pred_col: 'pred_future_rv_20d'}).assign(model=label)

pred_tables = []
for label, path in available:
    pred_tables.append(load_prediction_table(label, path))
    print(label, path, pred_tables[-1].shape)

display(pd.concat([t.head(3) for t in pred_tables], ignore_index=True))

## 3. Shared regime-layer function

In [ ]:
def build_forecast_regime_layer(pred_table):
    layer = timeline.merge(pred_table, on='date', how='inner')
    layer['forecast_high_vol'] = layer['pred_future_rv_20d'] > thresholds['high_vol_q80']
    layer['forecast_extreme_vol'] = layer['pred_future_rv_20d'] > thresholds['extreme_vol_q90']
    confirm_cols = ['flag_vix_level', 'flag_vix_range', 'flag_drawdown', 'flag_selloff', 'flag_volume']
    layer['confirm_score'] = layer[confirm_cols].sum(axis=1)
    layer['forecast_regime'] = 'normal'
    layer.loc[layer['forecast_high_vol'] | (layer['confirm_score'] >= 2), 'forecast_regime'] = 'watch'
    layer.loc[(layer['forecast_high_vol'] & (layer['confirm_score'] >= 1)) | layer['forecast_extreme_vol'], 'forecast_regime'] = 'warning'
    layer.loc[
        (layer['forecast_extreme_vol'] & (layer['confirm_score'] >= 2) & (layer['flag_drawdown'] | layer['flag_selloff'] | layer['flag_vix_level']))
        | (layer['forecast_high_vol'] & (layer['confirm_score'] >= 4) & (layer['flag_drawdown'] | layer['flag_vix_level'])),
        'forecast_regime',
    ] = 'crisis_like'
    return layer

layers = [build_forecast_regime_layer(t) for t in pred_tables]
counts = pd.concat([x.groupby(['model', 'forecast_regime', 'direction_tag']).size().reset_index(name='n') for x in layers], ignore_index=True)
counts.to_csv(out_table_dir / 'phase2_qrc_progress_regime_counts.csv', index=False)
counts

## 4. Forecast and regime metrics

In [ ]:
def flag_metrics(signal, event):
    signal = pd.Series(signal).astype(bool).to_numpy()
    event = pd.Series(event).astype(bool).to_numpy()
    tp = int(np.sum(signal & event))
    fp = int(np.sum(signal & ~event))
    fn = int(np.sum(~signal & event))
    tn = int(np.sum(~signal & ~event))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn, 'precision': precision, 'recall': recall, 'f1': f1, 'signal_rate': float(signal.mean())}

def forecast_metrics(layer):
    m = evaluate_volatility_forecast(layer['actual_future_rv_20d'], layer['pred_future_rv_20d'])
    return {
        'model': layer['model'].iloc[0],
        'rmse': m.rmse,
        'qlike': m.qlike,
        'mz_r2': m.mz_r2,
        'corr': float(np.corrcoef(layer['actual_future_rv_20d'], layer['pred_future_rv_20d'])[0, 1]),
        'pred_std': float(layer['pred_future_rv_20d'].std()),
    }

def regime_metrics(layer):
    signals = {
        'q80_watch_plus': ('future_high_vol_event_q80', layer['forecast_regime'].isin(['watch', 'warning', 'crisis_like'])),
        'q90_warning_plus': ('future_extreme_vol_event_q90', layer['forecast_regime'].isin(['warning', 'crisis_like'])),
        'q95_crisis_like': ('future_crisis_candidate_q95', layer['forecast_regime'].eq('crisis_like')),
    }
    row = {'model': layer['model'].iloc[0]}
    for name, (event, signal) in signals.items():
        met = flag_metrics(signal, layer[event])
        row[f'{name}_precision'] = met['precision']
        row[f'{name}_recall'] = met['recall']
        row[f'{name}_f1'] = met['f1']
    return row

forecast_summary = pd.DataFrame([forecast_metrics(x) for x in layers])
regime_summary = pd.DataFrame([regime_metrics(x) for x in layers])
progress_summary = forecast_summary.merge(regime_summary, on='model')
progress_summary.to_csv(out_table_dir / 'phase2_qrc_progress_regime_ablation_summary.csv', index=False)
progress_summary

## 5. Compact progress plot

In [ ]:
plot_cols = ['rmse', 'q80_watch_plus_f1', 'q90_warning_plus_f1', 'q95_crisis_like_f1']
plot_df = progress_summary.set_index('model')[plot_cols].copy()
display(plot_df)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(plot_cols))
width = 0.8 / len(plot_df)
for i, (model, row) in enumerate(plot_df.iterrows()):
    ax.bar(x + (i - (len(plot_df)-1)/2) * width, row.values, width=width, label=model)

ax.set_xticks(x)
ax.set_xticklabels(['RMSE', 'q80 watch+ F1', 'q90 warning+ F1', 'q95 crisis-like F1'])
ax.set_title('Reservoir progress: initial QRC → final QRC → ESN reference')
ax.set_ylabel('metric value')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(out_fig_dir / 'phase2_qrc_progress_regime_ablation.png', dpi=180)
plt.show()

## 6. Timeline comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
base = layers[0]
ax.plot(base['date'], base['actual_future_rv_20d'], label='actual future_rv_20d', linewidth=1.3)
for layer in layers:
    ax.plot(layer['date'], layer['pred_future_rv_20d'], label=layer['model'].iloc[0], linewidth=1.1)
ax.axhline(thresholds['high_vol_q80'], linestyle='--', label='train q80')
ax.axhline(thresholds['extreme_vol_q90'], linestyle='--', label='train q90')
ax.axhline(thresholds['crisis_candidate_q95'], linestyle='--', label='train q95')
ax.set_title('Forecast progression and tail-amplitude calibration')
ax.set_ylabel('future_rv_20d')
ax.legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(out_fig_dir / 'phase2_qrc_progress_forecast_timeline.png', dpi=180)
plt.show()

## Interpretation

Use this notebook only if the optional initial-QRC table is available and the comparison is visually useful. The intended message is: final QRC is a working, improved quantum-reservoir prototype; ESN is a strong classical reservoir reference showing the attainable ceiling and the remaining tail-calibration gap.